In [0]:
from pyspark.sql.functions import col, when, sum as spark_sum

# Read silver table
df = spark.table("workspace.default.silver_diabetic")

# Feature engineering
med_cols = ["metformin", "repaglinide", "nateglinide", "chlorpropamide",
            "glimepiride", "acetohexamide", "glipizide", "glyburide",
            "tolbutamide", "pioglitazone", "rosiglitazone", "acarbose",
            "miglitol", "troglitazone", "tolazamide", "examide",
            "citoglipton", "insulin", "glyburide-metformin",
            "glipizide-metformin", "glimepiride-pioglitazone",
            "metformin-rosiglitazone", "metformin-pioglitazone"]

# Count active medications
df = df.withColumn("num_active_meds",
    sum([when(col(c) != "No", 1).otherwise(0) for c in med_cols]))

# Insulin flag
df = df.withColumn("on_insulin",
    when(col("insulin") != "No", 1).otherwise(0))

# Medication changed
df = df.withColumn("med_changed",
    when(col("change") == "Ch", 1).otherwise(0))

# Total prior visits
df = df.withColumn("total_prior_visits",
    col("number_outpatient") + col("number_emergency") + col("number_inpatient"))

# High utilizer
df = df.withColumn("high_utilizer",
    when(col("total_prior_visits") >= 3, 1).otherwise(0))

# Age numeric
from pyspark.sql.functions import create_map, lit
from itertools import chain

age_mapping = {
    "[0-10)": 5, "[10-20)": 15, "[20-30)": 25, "[30-40)": 35,
    "[40-50)": 45, "[50-60)": 55, "[60-70)": 65, "[70-80)": 75,
    "[80-90)": 85, "[90-100)": 95
}
mapping_expr = create_map([lit(x) for x in chain(*age_mapping.items())])
df = df.withColumn("age_numeric", mapping_expr[col("age")])

# Target variable
df = df.withColumn("readmitted_binary",
    when(col("readmitted") == "NO", 0).otherwise(1))

# Save gold table
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_diabetic")

print("✅ Gold Delta table saved!")
print("Rows:", df.count())
print("Columns:", len(df.columns))

In [0]:
# Create reporting table for Power BI
reporting = spark.table("workspace.default.gold_diabetic")

# Select only columns needed for dashboards
reporting = reporting.select(
    "patient_nbr",
    "age",
    "age_numeric", 
    "race",
    "gender",
    "time_in_hospital",
    "num_lab_procedures",
    "num_medications",
    "num_active_meds",
    "on_insulin",
    "total_prior_visits",
    "high_utilizer",
    "number_diagnoses",
    "readmitted",
    "readmitted_binary"
)

# Save as reporting table
reporting.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.reporting_diabetic")

print("✅ Reporting table saved!")
print("Rows:", reporting.count())
print("Columns:", len(reporting.columns))

# Preview
reporting.show(5)